In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras

# Charger les données
df = pd.read_csv('posture_dataset.csv')
X = df.drop('label', axis=1).values
y = keras.utils.to_categorical(df['label'].values, 3)

# Normalisation
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Reshape pour Conv1D : (samples, timesteps, features)
X = X.reshape(X.shape[0], X.shape[1], 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Modèle CNN (comme Lab 1 UCI-HAR mais adapté)
model = keras.Sequential([
    keras.layers.Conv1D(32, 3, activation='relu', input_shape=(X.shape[1], 1)),
    keras.layers.MaxPooling1D(2),
    keras.layers.Conv1D(64, 3, activation='relu'),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(3, activation='softmax')  # vert / orange / rouge
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=50, validation_data=(X_test, y_test))

# Sauvegarder
import joblib
joblib.dump(scaler, 'posture_scaler.pkl')
model.save('posture_model.h5')
print("Accuracy:", model.evaluate(X_test, y_test)[1])

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import joblib

df = pd.read_csv('posture_dataset.csv')
X = df.drop('label', axis=1).values
y = df['label'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

acc = accuracy_score(y_test, rf.predict(X_test))
print(f"Accuracy : {acc*100:.1f}%")

joblib.dump(rf, 'posture_model.pkl')
joblib.dump(scaler, 'posture_scaler.pkl')
print("Fichiers sauvegardés !")

Accuracy : 98.4%
Fichiers sauvegardés !


In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

df = pd.read_csv('posture_dataset.csv')
print("Distribution originale :")
print(df['label'].value_counts())

# Équilibrage
df0 = df[df['label'] == 0]
df1 = df[df['label'] == 1]
df2 = df[df['label'] == 2]
n = max(len(df0), len(df1), len(df2))
df_balanced = pd.concat([
    resample(df0, replace=True, n_samples=n, random_state=42),
    resample(df1, replace=True, n_samples=n, random_state=42),
    resample(df2, replace=True, n_samples=n, random_state=42),
]).sample(frac=1, random_state=42)

print("\nAprès équilibrage :")
print(df_balanced['label'].value_counts())

X = df_balanced.drop('label', axis=1).values
y = df_balanced['label'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(f"\nAccuracy : {accuracy_score(y_test, y_pred)*100:.1f}%")
print("\nDétail par classe :")
print(classification_report(y_test, y_pred, target_names=['vert','orange','rouge']))

joblib.dump(rf, 'posture_model.pkl')
joblib.dump(scaler, 'posture_scaler.pkl')
print("\nFichiers sauvegardés !")

Distribution originale :
label
2    487
0    119
1     12
Name: count, dtype: int64

Après équilibrage :
label
1    487
2    487
0    487
Name: count, dtype: int64

Accuracy : 100.0%

Détail par classe :
              precision    recall  f1-score   support

        vert       1.00      1.00      1.00        97
      orange       1.00      1.00      1.00        98
       rouge       1.00      1.00      1.00        98

    accuracy                           1.00       293
   macro avg       1.00      1.00      1.00       293
weighted avg       1.00      1.00      1.00       293


Fichiers sauvegardés !
